# AGENTS4DQ — Agents for Data Quality

**REPLY × LUISS Machine Learning Project | A.Y. 2025/26**

| Member | Student ID |
|---|---|
| Maria Dichio | 822231 |
| Gianfranco Votta *(Captain)* | 807861 |
| Armando Fornario | 813811 |

---

The system is a multi-agent pipeline that automatically assesses the quality of a raw CSV dataset through four sequential stages: **Schema Validation → Completeness Analysis → Consistency Validation → Anomaly Detection**.

Each agent combines LLM-based reasoning with deterministic statistical checks.

## 0 — Environment Setup & Imports

We begin by installing the required packages and importing all dependencies.  
We also configure **LLM caching** via SQLite to avoid redundant API calls during repeated runs.

> **Note:** A `.env` file with `OPENROUTER_API_KEY` and `GOOGLE_API_KEY` must be present at the project root.

In [3]:
import sys
import os
import re
import json
import time
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from dotenv import load_dotenv
load_dotenv()

from langchain_openrouter import ChatOpenRouter
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.globals import set_llm_cache
from langchain_community.cache import SQLiteCache

# Enable LLM response caching
CACHE_DB_PATH = "notebook_cache.db"
set_llm_cache(SQLiteCache(database_path=CACHE_DB_PATH))

print("Environment ready.")
print(f"Python {sys.version}")
print(f"pandas {pd.__version__}, numpy {np.__version__}")

Environment ready.
Python 3.13.7 (v3.13.7:bcee1c32211, Aug 14 2025, 19:10:51) [Clang 16.0.0 (clang-1600.0.26.6)]
pandas 2.2.3, numpy 2.4.3


## 1 — Utility Functions

### 1.1 — `Outputs` helper class

The `Outputs` class wraps raw LLM responses and provides convenience methods to extract clean text or parsed JSON lists. It strips markdown code fences that some models occasionally include in their output.

In [4]:
class Outputs:
    """Wraps an LLM response and provides clean text / list extraction."""

    def __init__(self, response):
        self.response = response

    def __str__(self):
        content = self.response
        if isinstance(content, list):
            return content[0].get("text", "")
        return str(content)

    def get_list_out(self):
        return json.loads(str(self))

    def get_text(self):
        content = self.response
        if isinstance(content, list):
            text = content[0].get("text", "")
        else:
            text = str(content)
        # Strip markdown code fences
        text = re.sub(r'^```[a-zA-Z]*\n?', '', text)
        text = re.sub(r'\n?```$', '', text)
        return text.strip()

### 1.2 — `process_csv`: File Loading

This function takes a file path and returns a tuple `[status_message, DataFrame]`.  
It supports both `.csv` and `.xlsx` formats.

In [5]:
def process_csv(file_path):
    """Load a CSV or Excel file into a pandas DataFrame."""
    if file_path.endswith('.csv'):
        try:
            df = pd.read_csv(file_path)
            return [f"SUCCESS: Loaded CSV — {len(df)} rows, columns: {list(df.columns)}", df]
        except Exception as e:
            return [f"FAILURE: {e}", None]
    elif file_path.endswith('.xlsx'):
        try:
            df = pd.read_excel(file_path)
            return [f"SUCCESS: Loaded Excel — {len(df)} rows, columns: {list(df.columns)}", df]
        except Exception as e:
            return [f"FAILURE: {e}", None]
    return ["FAILURE: Unsupported file type. Please provide a .csv or .xlsx file.", None]

### 1.3 — `get_dataframe_patterns`: Regex Pattern Fingerprinting

This is a key utility used by the **Consistency Validator** and the **Anomaly Detector**.  
For every cell in the DataFrame, it converts the value into a *shape string*:
- All digits → `N`
- All letter sequences → `W`
- Punctuation and separators are preserved as-is

The result is a per-column frequency map of patterns. For example, a date column might show:  
`{"NN/NN/NNNN": 990, "NN-NN-NNNN": 10}` — immediately revealing a format inconsistency.

In [6]:
def get_dataframe_patterns(df):
    """Build a regex-based pattern frequency map for every column."""

    def get_shape(value):
        if pd.isna(value):
            return "NULL"
        s = str(value)
        s = re.sub(r'[a-zA-Z]+', 'W', s)
        s = re.sub(r'[0-9]', 'N', s)
        return s

    all_column_patterns = {}
    for col in df.columns:
        patterns = df[col].map(get_shape).value_counts().to_dict()
        all_column_patterns[col] = patterns
    return all_column_patterns

## 2 — Load the Dataset

We load the first test dataset (`spesa.csv`) — a NoiPA expenditure file with 18 columns covering tax categories, ministry descriptions, and spending amounts.

The output below will show the number of rows, the list of columns, and a preview of the first 5 rows so we can visually inspect the data before running the agents.

In [10]:
DATA_PATH = "data/raw/spesa.csv"

status, df = process_csv(DATA_PATH)
print(status)
df.head()

FAILURE: [Errno 2] No such file or directory: 'data/raw/spesa.csv'


AttributeError: 'NoneType' object has no attribute 'head'

Let's also take a quick look at data types and basic stats to get a feel for the dataset before the agents run.

In [ ]:
print("Shape:", df.shape)
print()
print("Dtypes:")
print(df.dtypes)
print()
print("Null counts:")
print(df.isnull().sum())

## 3 — Agent 1: Schema Validator

The Schema Validator performs two checks:

1. **Data Type Validation** — It sends the column names, inferred dtypes, and the first few rows to the LLM, asking it to identify columns where the data type does not match logical expectations (e.g., a `Price` column stored as `object`).
2. **Naming Convention Check** — It sends the column names to the LLM and asks it to flag any that violate common naming standards (e.g., special characters, inconsistent casing, reserved words).

We expect the LLM to flag columns like `ente%code`, `2cod_imposta`, `cod imposta ext`, and `Tipo Imposta` (mixed casing) in the `spesa.csv` dataset.

In [ ]:
class SchemaValidator:
    def __init__(self):
        self.model = ChatOpenRouter(model="stepfun/step-3.5-flash:free", temperature=0)

    def run_validation_check(self, df, dataset_name):
        internal_schema_info = {
            col: str(dtype) for col, dtype in zip(df.columns, df.dtypes)
        }
        prompt = (
            f"Context: {dataset_name}\n"
            f"Current Schema (Column: Type): {internal_schema_info}\n\n"
            f"Head of dataframe: {df.head().to_dict()}\n"
            "Task: Identify columns where the data type does not match logical expectations "
            "based on the column name and the head of the dataframe.\n"
            "(e.g., a 'Price' column being a string, or 'ID' being a float).\n\n"
            "Output Rules:\n"
            "1. If mismatches found: Begin with 'Here\'s a list of columns with type mismatches I found:' "
            "then list each with a bullet point and brief explanation.\n"
            "2. If all match: Return ONLY 'All columns have the expected types they should have.'"
        )
        return self.model.invoke(prompt).content

    def run_naming_check(self, df, dataset_name):
        prompt = (
            f"Context: {dataset_name}\n"
            f"Current column names: {df.columns.to_list()}\n\n"
            "Task: Identify columns whose name doesn't follow a known standard naming convention "
            "or the general structure of the other column names "
            "(e.g., special characters, wrong casing, reserved words).\n\n"
            "Output Rules:\n"
            "1. If issues found: Begin with 'Here\'s a list of columns with naming violations I found:' "
            "then list each with a bullet point.\n"
            "2. If no issues: Return ONLY 'No columns with naming violations found.'"
        )
        return self.model.invoke(prompt).content

schema_validator = SchemaValidator()
print("SchemaValidator initialised.")

### 3.1 — Run Data Type Validation

The cell below sends the schema and a sample of the data to the LLM.  
The output is a natural-language report listing any columns with suspicious types.

In [ ]:
type_check_result = schema_validator.run_validation_check(df, DATA_PATH)
print(type_check_result)

### 3.2 — Run Naming Convention Check

The cell below checks whether column names follow a consistent naming convention.  
We expect violations for columns like `ente%code`, `2cod_imposta`, `cod imposta ext`, `Tipo Imposta`, and `SPESA TOTALE`.

In [ ]:
naming_check_result = schema_validator.run_naming_check(df, DATA_PATH)
print(naming_check_result)

## 4 — Agent 2: Completeness Analyst

The Completeness Analyst works in three steps:

1. **Placeholder Detection** — The LLM inspects the head of the DataFrame and returns a list of common placeholder strings (e.g., `"-"`, `"null"`, `"N/A"`, `"unknown"`) that should be treated as missing values. These are then replaced with `NaN` in-place.
2. **Missing-Value Computation** — A deterministic function computes the percentage of `NaN` values per column and overall.
3. **Summarisation** — The LLM receives the statistics and produces a human-readable report, including a table of per-column completeness and a list of columns flagged for removal (>50% missing).

In [ ]:
class CompletenessAnalyst:
    def __init__(self):
        self.model = ChatOpenRouter(model="stepfun/step-3.5-flash:free", temperature=0)

    def run_completeness_analysis(self, df, dataset_name):
        """Detect placeholder strings via LLM and replace them with NaN."""
        prompt = (
            "You are a data completeness analyst. "
            "Identify common placeholder strings that represent missing values.\n"
            f"Context: {dataset_name}\n"
            f"Columns: {list(df.columns)}\n"
            f"Head: {df.head().to_dict()}\n"
            "Identify placeholders (like 'null', 'n/a', '-', 'none') in both English "
            "and the inferred language of the dataset.\n"
            'Output: Return ONLY a JSON list of strings. Example: [\"-\", \"null\", \"N/A\"]'
        )
        message = self.model.invoke(prompt)
        placeholders = json.loads(str(Outputs(message.content)))
        df.replace(placeholders, np.nan, inplace=True)
        return placeholders

    def NA_percentages(self, df):
        """Compute per-column missing-value percentages."""
        return {
            col: df[col].isnull().sum() / len(df)
            for col in df.columns
        }

    def summarize(self, schema, overall, to_drop, dataset_name):
        """Ask the LLM to produce a human-readable completeness report."""
        prompt = (
            f"Context: {dataset_name}\n"
            "The dataset has been analysed for completeness.\n"
            f"Per-column missing percentages: {schema}\n"
            f"Overall missing percentage: {overall}\n"
            f"Columns with >50% missing: {to_drop}\n"
            "Task: Summarise the findings in three sections:\n"
            "1. Overall Completeness Summary (bullet points)\n"
            "2. Column Completeness Highlights (table + critical insights)\n"
            "3. Flag columns for removal (>50% missing, with explanation)"
        )
        return self.model.invoke(prompt).content

completeness_analyst = CompletenessAnalyst()
print("CompletenessAnalyst initialised.")

### 4.1 — Detect Placeholders

The LLM examines the head of the dataset and returns a list of strings that should be treated as missing values. These are replaced with `NaN` in-place, so subsequent analysis correctly counts them.

In [ ]:
placeholders_found = completeness_analyst.run_completeness_analysis(df, DATA_PATH)
print("Placeholders detected and replaced:", placeholders_found)

### 4.2 — Compute Missing-Value Percentages

We now compute the percentage of null values for each column and overall. Columns exceeding 50% missing are flagged as candidates for removal.

In [ ]:
completeness_output = completeness_analyst.NA_percentages(df)

# Display as a readable table
completeness_df = pd.DataFrame.from_dict(
    completeness_output, orient="index", columns=["Missing %"]
).sort_values("Missing %", ascending=False)
completeness_df["Missing %"] = completeness_df["Missing %"].map(lambda x: f"{x:.2%}")
print(completeness_df.to_string())

overall_missing = sum(completeness_output.values()) / len(completeness_output)
droppable_columns = [col for col, v in completeness_output.items() if v > 0.5]

print(f"\nOverall missing: {overall_missing:.2%}")
print(f"Columns flagged for removal (>50% missing): {droppable_columns}")

### 4.3 — LLM Summary

The LLM synthesises the statistics above into a structured, human-readable report.

In [ ]:
completeness_summary = completeness_analyst.summarize(
    completeness_output, overall_missing, droppable_columns, DATA_PATH
)
print(Outputs(completeness_summary).get_text())

## 5 — Agent 3: Consistency Validator

The Consistency Validator performs three checks:

1. **Duplicate Detection** — Counts exact row duplicates. The LLM also identifies likely unique-identifier columns, and we check for duplicates on those keys specifically.
2. **Format Consistency** — Uses the regex pattern fingerprint (from Section 1.3) to flag columns where a small minority of values deviate from the dominant format.
3. **Cross-Column Logic** — Sends a sample of 30 rows to the LLM, which infers logical rules between columns (e.g., date ordering, mathematical sums, redundant mirror columns) and checks whether they hold.

In [ ]:
class ConsistencyValidator:
    def __init__(self):
        self.model = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite-preview", temperature=0)

    def run_duplicate_detection(self, df, dataset_name):
        """Detect exact duplicates and key-column duplicates."""
        exact_dupes = df.duplicated().sum()
        prompt = (
            f"Dataset: {dataset_name}\n"
            f"Columns: {list(df.columns)}\n"
            "Task: Identify columns that likely represent a unique identifier "
            '(e.g., "ID", "Tax Code", "Email").\n'
            'Output: Return ONLY a JSON list of column names. Example: [\"user_id\"]\n'
            "If none found, return []."
        )
        message = self.model.invoke(prompt)
        try:
            key_columns = json.loads(str(Outputs(message.content)))
        except json.JSONDecodeError:
            key_columns = []

        key_dupes_report = []
        for col in key_columns:
            if col in df.columns:
                dupes = df.duplicated(subset=[col]).sum()
                if dupes > 0:
                    key_dupes_report.append(f"Column '{col}': {dupes} duplicate entries")

        return {
            "exact_duplicates": exact_dupes,
            "key_columns_identified": key_columns,
            "key_column_duplicates": key_dupes_report if key_dupes_report else "No key duplicates found."
        }

    def run_format_consistency_check(self, pattern_report_str, df, dataset_name):
        """Flag columns with inconsistent value formats based on pattern fingerprints."""
        prompt = (
            f"Dataset: {dataset_name}\n"
            f"Columns: {list(df.columns)}\n"
            f"Regex pattern report: {pattern_report_str}\n\n"
            "TASK: Flag columns with inconsistent patterns (anomalous patterns appearing in <0.05% of rows).\n"
            "RULES:\n"
            "- Do NOT flag high-variety columns (e.g., descriptions).\n"
            "- ONLY flag if a clear dominant format is broken by a few outliers.\n\n"
            "Output:\n"
            "1. If inconsistencies found: List column + explanation.\n"
            "2. If consistent: Return ONLY 'Formats are consistent'."
        )
        return self.model.invoke(prompt).content

    def run_cross_column_logic(self, df_clean):
        """Infer and validate logical rules between columns on a sample."""
        sample_size = min(30, len(df_clean))
        sample_data = df_clean.sample(n=sample_size, random_state=42).to_markdown()
        prompt = (
            f"You are a Senior Data Quality Analyst. Here is a sample of {sample_size} rows.\n\n"
            f"### DATA SAMPLE:\n{sample_data}\n\n"
            "### TASK:\n"
            "1. Analyze relationships between columns and infer logical rules.\n"
            "2. Audit the sample against these rules.\n"
            "3. Identify redundant/mirror columns.\n\n"
            "### OUTPUT (Strict Markdown):\n"
            "**Inferred Logical Rules:** - [rule]\n"
            "**Data Redundancy Alerts:** - [col A] and [col B] appear identical.\n\n"
            "If NO issues found: 'No logical violations detected in sample.'"
        )
        return self.model.invoke(prompt).content

consistency_validator = ConsistencyValidator()
print("ConsistencyValidator initialised.")

### 5.1 — Duplicate Detection

We first count exact row duplicates, then the LLM identifies likely key columns and we check for duplicates on those specifically. The output shows the counts and any flagged key-column duplicates.

In [ ]:
duplicate_results = consistency_validator.run_duplicate_detection(df, DATA_PATH)

print(f"Exact row duplicates: {duplicate_results['exact_duplicates']}")
print(f"Key columns identified: {duplicate_results['key_columns_identified']}")
print(f"Key column duplicates: {duplicate_results['key_column_duplicates']}")

### 5.2 — Format Consistency Check

We compute the regex pattern fingerprint for every column, then send it to the LLM.  
The LLM flags columns where a dominant pattern is broken by a small number of outlier formats.

The pattern legend is: `N` = digit, `W` = word/letter sequence.

In [ ]:
pattern_report = get_dataframe_patterns(df)

# Show pattern report for a few columns as example
for col in list(pattern_report.keys())[:4]:
    print(f"\n{col}:")
    for pattern, count in sorted(pattern_report[col].items(), key=lambda x: -x[1])[:5]:
        print(f"  {pattern}: {count}")

pattern_report_str = json.dumps(pattern_report, indent=2)

*Continued from above:*

In [ ]:
format_results = consistency_validator.run_format_consistency_check(pattern_report_str, df, DATA_PATH)
print(Outputs(format_results).get_text())

### 5.3 — Cross-Column Logic Validation

The LLM receives a sample of 30 rows and infers logical rules that *should* hold between columns (e.g., `spesa` ≡ `SPESA TOTALE`, `cod_imposta` ≡ `2cod_imposta`). It then audits the sample against these rules and reports violations or redundancies.

Before sampling, we drop the columns previously flagged as too sparse (>50% missing) and remove rows with remaining nulls to give the LLM a clean sample.

In [ ]:
df_clean = df.drop(columns=droppable_columns, errors='ignore').dropna(axis=0)
print(f"Clean sample shape (after dropping sparse cols + null rows): {df_clean.shape}")

logic_results = consistency_validator.run_cross_column_logic(df_clean)
print(Outputs(logic_results).get_text())

## 6 — Agent 4: Anomaly Detector

The Anomaly Detector performs two types of detection:

1. **Univariate Outlier Detection** — The LLM selects numerical columns that are meaningful for outlier analysis (e.g., `spesa` but not `ente`). For each selected column, we apply the **3σ rule**: values more than 3 standard deviations from the mean are flagged as outliers.

2. **Categorical Anomaly Detection** — The LLM selects categorical columns suitable for frequency analysis (e.g., `tipo_imposta` but not `descrizione`). For each selected column, values appearing in fewer than 1% of rows are flagged as rare/anomalous categories.

In [ ]:
class AnomalyDetector:
    def __init__(self):
        self.model = ChatOpenRouter(model="stepfun/step-3.5-flash:free", temperature=0)

    def univariate_outlier_detection(self, pattern_report_str, df, dataset_name):
        """LLM selects numerical columns, then 3-sigma outlier detection is applied."""
        prompt = (
            f"Context: {dataset_name}\n"
            f"Sample: {df.head().to_dict()}\n"
            f"Pattern report: {pattern_report_str}\n"
            "Task: Identify candidate columns for univariate outlier detection.\n"
            "Conditions: (1) mostly numeric patterns, (2) semantically meaningful for outliers "
            "(e.g., 'expenses' yes, 'id' no).\n"
            'Output: ONLY a JSON list. Example: [\"spesa\"]'
        )
        message = self.model.invoke(prompt)
        univariate_columns = json.loads(str(Outputs(message.content)))

        report_lines = []
        if not univariate_columns:
            report_lines.append("No columns were found as candidates for univariate outlier detection.")
        else:
            report_lines.append(f"Candidate columns: {univariate_columns}\n")
            for col in univariate_columns:
                if col not in df.columns:
                    continue
                series = pd.to_numeric(df[col], errors='coerce').dropna()
                if series.empty:
                    continue
                mean_val, std_val = series.mean(), series.std()
                upper, lower = mean_val + 3 * std_val, mean_val - 3 * std_val
                outliers = series[(series > upper) | (series < lower)]
                if len(outliers) == 0:
                    report_lines.append(f"Column '{col}': No outliers detected (3σ).")
                else:
                    report_lines.append(f"Column '{col}': {len(outliers)} outlier(s) found.")
                    report_lines.append(f"  Mean={mean_val:.2f}, Std={std_val:.2f}, Bounds=[{lower:.2f}, {upper:.2f}]")
                    report_lines.append(f"  Outlier values: {outliers.to_dict()}")
                report_lines.append("")

        return "\n".join(report_lines), univariate_columns

    def categorical_outlier_detection(self, pattern_report_str, df, dataset_name):
        """LLM selects categorical columns, then rare-value detection is applied."""
        prompt = (
            f"Context: {dataset_name}\n"
            f"Sample: {df.head().to_dict()}\n"
            f"Pattern report: {pattern_report_str}\n"
            "Task: Identify candidate columns for categorical outlier detection.\n"
            "Conditions: (1) mostly word-based patterns, (2) semantically meaningful for "
            "frequency analysis (e.g., 'job_title' yes, 'notes' no).\n"
            'Output: ONLY a JSON list. Example: [\"tipo_imposta\"]'
        )
        message = self.model.invoke(prompt)
        categorical_columns = json.loads(str(Outputs(message.content)))

        report_lines = []
        if not categorical_columns:
            report_lines.append("No columns were found as candidates for categorical outlier detection.")
        else:
            report_lines.append(f"Candidate columns: {categorical_columns}\n")
            for col in categorical_columns:
                if col not in df.columns:
                    continue
                series = df[col].astype(str)
                counts = series.value_counts()
                rare = counts[counts < len(series) * 0.01]
                if len(rare) == 0:
                    report_lines.append(f"Column '{col}': No rare categories (<1%).")
                else:
                    report_lines.append(f"Column '{col}': {len(rare)} rare category/ies (<1% frequency).")
                    report_lines.append(f"  Rare values: {rare.to_dict()}")
                report_lines.append("")

        return "\n".join(report_lines), categorical_columns

anomaly_detector = AnomalyDetector()
print("AnomalyDetector initialised.")

### 6.1 — Univariate Outlier Detection

The LLM identifies which numerical columns are appropriate for outlier analysis. Then, for each candidate column, we compute the mean and standard deviation and flag values beyond ±3σ.

The output shows the candidate columns selected by the LLM and, for each, the number of outliers and their values.

In [ ]:
outlier_report, outlier_cols = anomaly_detector.univariate_outlier_detection(
    pattern_report_str, df, DATA_PATH
)
print(outlier_report)

### 6.2 — Categorical Anomaly Detection

Similarly, the LLM selects categorical columns where frequency analysis is meaningful. Values appearing in fewer than 1% of rows are reported as rare/anomalous categories.

In [ ]:
cat_report, cat_cols = anomaly_detector.categorical_outlier_detection(
    pattern_report_str, df, DATA_PATH
)
print(cat_report)

## 7 — Experiment 2: `attivazioniCessazioni.csv`

To validate that the system generalises to structurally different datasets, we now run the full pipeline on a second NoiPA file containing personnel activation/termination records.

This dataset has different columns (`mese`, `anno`, `attivazioni`, `cessazioni`, `qualifica`, etc.) and different types of intentional noise (e.g., `3descrizione`, `regione%sede`, `att ivazioni`, casing inconsistencies in `provincia_sede`).

In [ ]:
DATA_PATH_2 = "data/attivazioniCessazioni.csv"

status2, df2 = process_csv(DATA_PATH_2)
print(status2)
df2.head()

### 7.1 — Schema Validation on Dataset 2

In [ ]:
print("=== Data Type Validation ===")
print(schema_validator.run_validation_check(df2, DATA_PATH_2))
print()
print("=== Naming Convention Check ===")
print(schema_validator.run_naming_check(df2, DATA_PATH_2))

### 7.2 — Completeness Analysis on Dataset 2

In [ ]:
placeholders2 = completeness_analyst.run_completeness_analysis(df2, DATA_PATH_2)
print("Placeholders detected:", placeholders2)

completeness2 = completeness_analyst.NA_percentages(df2)
overall2 = sum(completeness2.values()) / len(completeness2)
droppable2 = [col for col, v in completeness2.items() if v > 0.5]

comp_df2 = pd.DataFrame.from_dict(completeness2, orient="index", columns=["Missing %"])
comp_df2 = comp_df2.sort_values("Missing %", ascending=False)
comp_df2["Missing %"] = comp_df2["Missing %"].map(lambda x: f"{x:.2%}")
print(comp_df2.to_string())
print(f"\nOverall: {overall2:.2%}")
print(f"Flagged for removal: {droppable2}")

*Continued from above:*

In [ ]:
summary2 = completeness_analyst.summarize(completeness2, overall2, droppable2, DATA_PATH_2)
print(Outputs(summary2).get_text())

### 7.3 — Consistency Validation on Dataset 2

In [ ]:
dup2 = consistency_validator.run_duplicate_detection(df2, DATA_PATH_2)
print(f"Exact duplicates: {dup2['exact_duplicates']}")
print(f"Key columns: {dup2['key_columns_identified']}")
print(f"Key dupes: {dup2['key_column_duplicates']}")

*Continued from above:*

In [ ]:
pattern2 = get_dataframe_patterns(df2)
pattern2_str = json.dumps(pattern2, indent=2)

fmt2 = consistency_validator.run_format_consistency_check(pattern2_str, df2, DATA_PATH_2)
print(Outputs(fmt2).get_text())

*Continued from above:*

In [ ]:
df2_clean = df2.drop(columns=droppable2, errors='ignore').dropna(axis=0)
print(f"Clean sample shape: {df2_clean.shape}")

logic2 = consistency_validator.run_cross_column_logic(df2_clean)
print(Outputs(logic2).get_text())

### 7.4 — Anomaly Detection on Dataset 2

In [ ]:
out2, out2_cols = anomaly_detector.univariate_outlier_detection(pattern2_str, df2, DATA_PATH_2)
print(out2)

*Continued from above:*

In [ ]:
cat2, cat2_cols = anomaly_detector.categorical_outlier_detection(pattern2_str, df2, DATA_PATH_2)
print(cat2)

## 8 — Summary & Conclusions

The AGENTS4DQ pipeline successfully processes both NoiPA datasets, detecting:

- **Schema issues**: type mismatches and naming convention violations
- **Completeness gaps**: placeholder strings, sparse columns, per-column/overall missing rates
- **Consistency problems**: exact and key-column duplicates, format inconsistencies, redundant mirror columns, cross-column logic violations
- **Anomalies**: statistical outliers in numerical columns and rare categories in categorical columns

The LLM-driven column selection makes the system **dataset-agnostic** — no hard-coded rules need to change when switching from `spesa.csv` to `attivazioniCessazioni.csv`.

**Limitations**: Sequential agent execution, no automated dataset correction, no global reliability score yet.  
**Future work**: Parallel execution, a dedicated Remediation Agent, a composite Reliability Score, and support for JSON/database inputs.